*0.2 Math / ML basics*

# sampling: top-p

**The situation.** Top-k=40 was set for the chatbot. For most tokens it works. But when the model is *certain* — the next token is "the" at 98% — 39 junk tokens stay eligible; and when the model is writing a list of possible names and 300 are reasonable, 260 good ones are cut. A fixed count is the wrong tool.

**Top-p (nucleus sampling).** Sort tokens by probability and keep the smallest set whose probabilities add up to *p* (say 0.9). When the model is certain, that is 1 token. When it is open, it can be hundreds. The cut adapts to the shape of the distribution. Available on OpenAI, Ollama, vLLM — everywhere.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The math.** Cumulative sum along the sorted probabilities, cut at p. Two distributions: a certain one and an open one.

In [2]:
import torch


def nucleus_size(probabilities: torch.Tensor, p: float) -> int:
    sorted_probabilities = probabilities.sort(descending=True).values
    cumulative = torch.cumsum(sorted_probabilities, dim=0)
    return int((cumulative < p).sum()) + 1  # tokens needed to reach p


certain = torch.softmax(torch.tensor([8.0, 2.0, 1.0, 0.5, 0.0]), dim=0)
open_ended = torch.softmax(torch.tensor([1.0, 1.0, 0.9, 0.9, 0.8]), dim=0)
for name, distribution in (
    ("certain distribution   ", certain),
    ("open-ended distribution", open_ended),
):
    shares = []
    for value in distribution.tolist():
        shares.append(f"{value:.2f}")
    print(f"{name}: {shares} → top-p 0.9 keeps {nucleus_size(distribution, 0.9)} of 5")
assert nucleus_size(certain, 0.9) == 1 and nucleus_size(open_ended, 0.9) == 5

certain distribution   : ['1.00', '0.00', '0.00', '0.00', '0.00'] → top-p 0.9 keeps 1 of 5
open-ended distribution: ['0.22', '0.22', '0.20', '0.20', '0.18'] → top-p 0.9 keeps 5 of 5


**Reading the output.** The same p=0.9 kept 1 token when the model was sure and all 5 when it was not. That is the adaptation top-k could not do.

**For real, on OpenAI.** `top_p=0.1` (almost greedy) vs `top_p=1.0` (everything), five samples each at temperature 1.

In [3]:
from openai import OpenAI

client = OpenAI(timeout=30)
counts = {}
for top_p in (0.1, 1.0):
    seen = set()
    for _ in range(5):
        reply = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "Name one colour. One word."}],
            max_tokens=3,
            temperature=1.0,
            top_p=top_p,
        )
        seen.add(reply.choices[0].message.content.strip().lower().strip("."))
    counts[top_p] = len(seen)
    print(f"top_p={top_p}: {len(seen)} distinct answer(s) → {sorted(seen)}")
assert counts[0.1] <= counts[1.0]

top_p=0.1: 1 distinct answer(s) → ['blue']


top_p=1.0: 2 distinct answer(s) → ['azure', 'blue']


```
sorted probabilities   0.55  0.20  0.10  0.06  0.04  0.02  0.01 …
cumulative             0.55  0.75  0.85  0.91 | p = 0.9 reached → keep 4, drop the rest
```

**The rule to remember.** Top-p keeps "enough tokens to cover p of the probability". 0.9–0.95 is the usual range. Set either temperature or top-p, not both to extreme values.

| Use it when | Don't when | Instead use |
|---|---|---|
| general chat and writing; any hosted API | you want a hard count guarantee for safety reasons | top-k, or both together |

**Watch out**
- `top_p=1.0` is *off*. Lowering it reduces variety; raising it above 1 does nothing.
- With temperature 0, top-p is irrelevant — the top token always wins.
- A very flat distribution at p=0.95 can still include hundreds of weak tokens; min-p (next) handles that case better.